# ⚡ StreamDiffusion Colab Server for TouchDesigner
This notebook runs **StreamDiffusion (SD-Turbo)** accelerated on Google Colab Pro GPU and bridges real-time streaming directly to your local **TouchDesigner** via a secure Cloudflare Tunnel.

### 🌟 Features Supported:
- **Dual-Prompt Latent Morphing**: Blend smoothly between Prompt A and Prompt B using real-time GPU latent interpolation.
- **Dynamic Denoise Strength**: Interactive 0.0 - 1.0 slider controlling diffusion timestep limits.
- **Audio-Reactivity**: CHOP connector and beat detection for music-synchronized visuals and random seed triggers.
- **Side-by-Side & Split Wipe**: Live comparison views between webcam/noise input and AI output.
- **High-Res Snapshots**: Save PNG frames with complete JSON parameter and telemetry sidecars.
- **Remote GPU Telemetry**: Live VRAM and GPU temperature reporting sent back to TouchDesigner.

### Instructions:
1. Make sure your runtime has a GPU: Go to **Runtime -> Change runtime type -> Hardware accelerator: GPU** (Select **A100**, **L4**, or **T4**).
2. Click **Runtime -> Run all** (or run each cell sequentially).
3. Cell 4 will warm up the pipeline and print your **TouchDesigner WebSocket URL** (`wss://....trycloudflare.com/ws/stream`).

In [ ]:
# Step 1: Check GPU
!nvidia-smi

In [ ]:
# Step 2: Clone StreamDiffusion & Install Compatible Dependencies
import os
print('Setting up StreamDiffusion environment...')

# Clone repo if not already present
if not os.path.exists('StreamDiffusion'):
    !git clone https://github.com/cumulo-autumn/StreamDiffusion.git

%cd StreamDiffusion
!pip install -q -e .
# Pin compatible versions and remove conflicting peft
!pip install -q 'transformers<4.40.0' 'huggingface-hub<0.26.0'
!pip uninstall -y peft || true
!pip install -q diffusers accelerate fastapi uvicorn websockets python-multipart opencv-python-headless pillow

# Download cloudflared for tunnel
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

print('✅ Setup complete!')

In [ ]:
# Step 3: Write server.py
server_code = "import os\nimport sys\nimport traceback\n\n# 1. Compatibility shims for huggingface_hub\nimport huggingface_hub\nimport huggingface_hub.constants\n\nif not hasattr(huggingface_hub.constants, 'hf_cache_home'):\n    huggingface_hub.constants.hf_cache_home = getattr(\n        huggingface_hub.constants, 'HF_HOME', os.path.expanduser('~/.cache/huggingface')\n    )\n\nif not hasattr(huggingface_hub, 'HfFolder'):\n    class HfFolder:\n        @classmethod\n        def get_token(cls):\n            try:\n                from huggingface_hub import get_token\n                return get_token()\n            except Exception:\n                return None\n        @classmethod\n        def save_token(cls, token):\n            pass\n    huggingface_hub.HfFolder = HfFolder\n\nif not hasattr(huggingface_hub, 'cached_download'):\n    from huggingface_hub import hf_hub_download\n    huggingface_hub.cached_download = hf_hub_download\n\n# 2. Compatibility shims for transformers\nimport transformers\nimport transformers.utils\n\nif not hasattr(transformers.utils, 'FLAX_WEIGHTS_NAME'):\n    transformers.utils.FLAX_WEIGHTS_NAME = 'flax_model.msgpack'\nif not hasattr(transformers.utils, 'FLAX_WEIGHTS_INDEX_NAME'):\n    transformers.utils.FLAX_WEIGHTS_INDEX_NAME = 'flax_model.msgpack.index.json'\n\nif not hasattr(transformers, 'EncoderDecoderCache'):\n    class EncoderDecoderCache:\n        pass\n    transformers.EncoderDecoderCache = EncoderDecoderCache\n    try:\n        import transformers.cache_utils\n        transformers.cache_utils.EncoderDecoderCache = EncoderDecoderCache\n    except Exception:\n        pass\n\nimport time\nimport asyncio\nimport json\nimport logging\nimport cv2\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom fastapi import FastAPI, WebSocket, WebSocketDisconnect\nfrom fastapi.responses import HTMLResponse\n\n# Disable autograd globally for fast inference\ntorch.set_grad_enabled(False)\nif torch.cuda.is_available():\n    torch.backends.cuda.matmul.allow_tf32 = True\n    torch.backends.cudnn.allow_tf32 = True\n    torch.backends.cudnn.benchmark = True\n\nsys.path.append('.')\ntry:\n    from utils.wrapper import StreamDiffusionWrapper\n    from streamdiffusion.image_utils import postprocess_image\nexcept ImportError:\n    StreamDiffusionWrapper = None\n    postprocess_image = None\n\n# Patch image_utils.pt_to_numpy just in case\ntry:\n    import streamdiffusion.image_utils\n    orig_pt_to_numpy = streamdiffusion.image_utils.pt_to_numpy\n    def safe_pt_to_numpy(images):\n        return images.detach().cpu().permute(0, 2, 3, 1).float().numpy()\n    streamdiffusion.image_utils.pt_to_numpy = safe_pt_to_numpy\nexcept Exception:\n    pass\n\nlogging.basicConfig(level=logging.INFO)\nlogger = logging.getLogger(\"StreamDiffusionServer\")\napp = FastAPI()\n\nstream_txt2img = None\nstream_img2img = None\npipeline_ready = False\n\n# Engine acceleration state\nenable_tensorrt = os.environ.get(\"ENABLE_TENSORRT\", \"0\") in (\"1\", \"true\", \"True\") or \"--tensorrt\" in sys.argv\nactive_engine = \"TensorRT (Turbo)\" if enable_tensorrt else \"PyTorch (FP16)\"\n\n# Prompt state\ncurrent_prompt_a = \"cinematic cyberpunk cityscape, neon lights, 8k, detailed\"\ncurrent_prompt_b = \"bioluminescent alien jungle, glowing flora, ethereal, 8k\"\ncurrent_prompt_blend = 0.0\ncurrent_neg_prompt = \"blurry, low quality, distorted, watermark\"\n\ncurrent_mode = \"txt2img\"\ncurrent_seed = 42\ncurrent_strength = 0.6\ncurrent_steps = 2\n\ndef get_t_list_for_steps(steps, mode=\"txt2img\", strength=0.6):\n    s = int(steps)\n    if mode == \"txt2img\":\n        if s == 1:\n            return [45]\n        elif s == 4:\n            return [0, 16, 32, 45]\n        else: # 2 steps (balanced default)\n            return [32, 45]\n    else: # img2img\n        t0 = max(5, min(44, int(45 - strength * 40)))\n        if s == 1:\n            return [t0]\n        elif s == 4:\n            t1 = int(t0 + (45 - t0) * 0.33)\n            t2 = int(t0 + (45 - t0) * 0.66)\n            return [t0, t1, t2, 45]\n        else: # 2 steps\n            return [t0, 45]\n\ndef apply_steps(new_steps):\n    global current_steps, stream_txt2img, stream_img2img\n    try:\n        s = int(new_steps)\n        if s not in (1, 2, 4):\n            s = 2\n        current_steps = s\n        \n        t_list_txt = get_t_list_for_steps(current_steps, \"txt2img\")\n        if stream_txt2img is not None and hasattr(stream_txt2img, \"stream\"):\n            stream_txt2img.stream.t_list = t_list_txt\n            stream_txt2img.prepare(prompt=current_prompt_a, num_inference_steps=50)\n            \n        t_list_img = get_t_list_for_steps(current_steps, \"img2img\", current_strength)\n        if stream_img2img is not None and hasattr(stream_img2img, \"stream\"):\n            stream_img2img.stream.t_list = t_list_img\n            stream_img2img.prepare(prompt=current_prompt_a, num_inference_steps=50)\n            \n        logger.info(f\"Updated diffusion steps to {current_steps}: txt2img={t_list_txt}, img2img={t_list_img}\")\n    except Exception as e:\n        logger.error(f\"Error applying steps: {e}\")\n\ndef set_seed(s):\n    torch.manual_seed(s)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(s)\n\ndef apply_prompts_and_blend(prompt_a, prompt_b=\"\", blend=0.0, neg_prompt=\"\"):\n    global current_prompt_a, current_prompt_b, current_prompt_blend, current_neg_prompt\n    if prompt_a is not None and str(prompt_a).strip():\n        current_prompt_a = str(prompt_a).strip()\n    if prompt_b is not None:\n        current_prompt_b = str(prompt_b).strip()\n    if blend is not None:\n        current_prompt_blend = max(0.0, min(1.0, float(blend)))\n    if neg_prompt is not None:\n        current_neg_prompt = str(neg_prompt).strip()\n\n    if not pipeline_ready or stream_txt2img is None:\n        return\n\n    try:\n        device = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n        core_pipe = stream_txt2img.stream.pipe\n\n        with torch.no_grad():\n            embed_a = core_pipe.encode_prompt(\n                prompt=current_prompt_a,\n                device=device,\n                num_images_per_prompt=1,\n                do_classifier_free_guidance=False,\n                negative_prompt=current_neg_prompt,\n            )[0]\n\n            if current_prompt_b and current_prompt_blend > 0.001:\n                embed_b = core_pipe.encode_prompt(\n                    prompt=current_prompt_b,\n                    device=device,\n                    num_images_per_prompt=1,\n                    do_classifier_free_guidance=False,\n                    negative_prompt=current_neg_prompt,\n                )[0]\n                alpha = current_prompt_blend\n                blended = (1.0 - alpha) * embed_a + alpha * embed_b\n            else:\n                blended = embed_a\n\n            # Apply blended embedding tensor to both pipelines\n            for s in [stream_txt2img, stream_img2img]:\n                if s is not None:\n                    target = s.stream if hasattr(s, \"stream\") else s\n                    if hasattr(target, \"prompt_embeds\"):\n                        target.prompt_embeds = blended.repeat(target.batch_size, 1, 1)\n\n        logger.info(f\"Prompts updated: A='{current_prompt_a[:25]}...', B='{current_prompt_b[:25]}...', blend={current_prompt_blend:.2f}\")\n    except Exception as e:\n        logger.error(f\"Error in prompt blending: {traceback.format_exc()}\")\n\ndef apply_strength(new_strength):\n    global current_strength, stream_img2img\n    current_strength = float(new_strength)\n    try:\n        if stream_img2img is not None:\n            if hasattr(stream_img2img, \"update_strength\"):\n                stream_img2img.update_strength(current_strength)\n            elif hasattr(stream_img2img, \"stream\") and hasattr(stream_img2img.stream, \"t_list\"):\n                new_t_list = get_t_list_for_steps(current_steps, \"img2img\", current_strength)\n                stream_img2img.stream.t_list = new_t_list\n                stream_img2img.prepare(prompt=current_prompt_a, num_inference_steps=50)\n                logger.info(f\"Updated img2img denoise strength to {current_strength:.2f} (t_index_list={new_t_list})\")\n    except Exception as e:\n        logger.error(f\"Error updating strength: {e}\")\n\ndef get_gpu_telemetry():\n    vram_str = \"--\"\n    temp_str = \"--\"\n    if torch.cuda.is_available():\n        try:\n            allocated = torch.cuda.memory_allocated() / (1024**3)\n            total = torch.cuda.get_device_properties(0).total_memory / (1024**3)\n            pct = (allocated / total) * 100\n            vram_str = f\"{allocated:.1f}/{total:.1f}GB ({pct:.0f}%)\"\n        except Exception:\n            pass\n        try:\n            import subprocess\n            out = subprocess.check_output(\n                [\"nvidia-smi\", \"--query-gpu=temperature.gpu\", \"--format=csv,noheader,nounits\"],\n                timeout=1,\n                text=True\n            ).strip()\n            temp_str = f\"{out}\u00b0C\"\n        except Exception:\n            pass\n    return vram_str, temp_str\n\ndef run_model_inference(input_image=None):\n    \"\"\"\n    Runs StreamDiffusion inference under torch.no_grad() inside the executor thread.\n    Never use torch.inference_mode() because StreamDiffusion caches tensors between steps.\n    \"\"\"\n    with torch.no_grad():\n        if current_mode in (\"img2img\", \"v2v\") and input_image is not None and stream_img2img is not None:\n            return stream_img2img(input_image)\n        if stream_txt2img is not None:\n            return stream_txt2img()\n        return None\n\ndef tensor_to_bgr(t):\n    \"\"\"\n    Zero-copy GPU post-processor:\n    Performs RGB->BGR reordering, clamping [0, 1] -> [0, 255] uint8 conversion\n    DIRECTLY in CUDA VRAM before transferring a contiguous uint8 buffer to CPU.\n    Saves 5-8ms of CPU latency compared to PIL/NumPy pipelines.\n    \"\"\"\n    if isinstance(t, Image.Image):\n        return cv2.cvtColor(np.array(t), cv2.COLOR_RGB2BGR)\n    if hasattr(t, \"detach\"):\n        t = t.detach()\n        if t.ndim == 4:\n            t = t[0]\n        t_bgr = t[[2, 1, 0], :, :]\n        return (t_bgr.permute(1, 2, 0).clamp(0, 1).mul(255).to(torch.uint8).cpu().numpy())\n    return np.array(t)\n\ndef compress_frame(arr_bgr, max_bytes=48000):\n    \"\"\"\n    Fast single-pass JPEG compression at Q55 (typically 22KB-35KB for 512x512).\n    Avoids multi-pass CPU overhead.\n    \"\"\"\n    ret, buf = cv2.imencode('.jpg', arr_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), 55])\n    if ret and len(buf) <= max_bytes:\n        return buf.tobytes()\n    h, w = arr_bgr.shape[:2]\n    small = cv2.resize(arr_bgr, (int(w * 0.8), int(h * 0.8)), interpolation=cv2.INTER_AREA)\n    ret, buf = cv2.imencode('.jpg', small, [int(cv2.IMWRITE_JPEG_QUALITY), 40])\n    return buf.tobytes() if ret else b\"\"\n\ndef load_pipeline():\n    global stream_txt2img, stream_img2img, pipeline_ready, active_engine\n    device = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    accel_mode = \"tensorrt\" if enable_tensorrt else \"none\"\n    print(f\">>> Initializing StreamDiffusion (SD-Turbo) on {device.upper()} with acceleration='{accel_mode}' (Engine: {active_engine})...\")\n\n    # 1. Text-to-Image pipeline: dynamic t_index_list based on current_steps\n    t_init_txt = get_t_list_for_steps(current_steps, \"txt2img\")\n    print(f\">>> Initializing Text-to-Image pipeline (t_index_list={t_init_txt}, steps={current_steps})...\")\n    try:\n        stream_txt2img = StreamDiffusionWrapper(\n            model_id_or_path=\"stabilityai/sd-turbo\",\n            t_index_list=t_init_txt,\n            frame_buffer_size=1,\n            width=512,\n            height=512,\n            warmup=5,\n            acceleration=accel_mode,\n            engine_dir=\"engines\",\n            mode=\"txt2img\",\n            output_type=\"pt\",\n            use_lcm_lora=False,\n            use_tiny_vae=True,\n            cfg_type=\"none\",\n            use_denoising_batch=True,\n            device=device,\n            seed=current_seed,\n        )\n    except Exception as e:\n        if accel_mode == \"tensorrt\":\n            print(f\">>> TensorRT initialization failed ({e}), falling back to PyTorch Native mode...\")\n            accel_mode = \"none\"\n            active_engine = \"PyTorch (FP16)\"\n            stream_txt2img = StreamDiffusionWrapper(\n                model_id_or_path=\"stabilityai/sd-turbo\",\n                t_index_list=t_init_txt,\n                frame_buffer_size=1,\n                width=512,\n                height=512,\n                warmup=5,\n                acceleration=\"none\",\n                mode=\"txt2img\",\n                output_type=\"pt\",\n                use_lcm_lora=False,\n                use_tiny_vae=True,\n                cfg_type=\"none\",\n                use_denoising_batch=True,\n                device=device,\n                seed=current_seed,\n            )\n        else:\n            raise\n    set_seed(current_seed)\n\n    # Enable Tensor Core channels_last memory format optimization\n    if device == \"cuda\" and stream_txt2img is not None and hasattr(stream_txt2img, \"stream\"):\n        try:\n            core_pipe = stream_txt2img.stream.pipe\n            if hasattr(core_pipe, \"unet\"):\n                core_pipe.unet.to(memory_format=torch.channels_last)\n            if hasattr(core_pipe, \"vae\") and core_pipe.vae:\n                core_pipe.vae.to(memory_format=torch.channels_last)\n            logger.info(\"Activated channels_last memory format for Tensor Core acceleration.\")\n        except Exception:\n            pass\n\n    # 2. Image-to-Image / StreamV2V pipeline: initial t_index_list based on current_steps and strength\n    t_init_img = get_t_list_for_steps(current_steps, \"img2img\", current_strength)\n    print(f\">>> Initializing Image-to-Image / StreamV2V pipeline (t_index_list={t_init_img}, strength={current_strength:.2f})...\")\n    try:\n        from streamdiffusion import StreamDiffusion\n        core_pipe = stream_txt2img.stream.pipe\n        stream_core = StreamDiffusion(\n            pipe=core_pipe,\n            t_index_list=t_init_img,\n            torch_dtype=core_pipe.dtype,\n            width=512,\n            height=512,\n            frame_buffer_size=1,\n            use_denoising_batch=True,\n            cfg_type=\"none\",\n        )\n        if hasattr(stream_txt2img.stream, \"vae\") and stream_txt2img.stream.vae:\n            stream_core.vae = stream_txt2img.stream.vae\n\n        class SharedImg2ImgWrapper:\n            def __init__(self, core):\n                self.stream = core\n            def prepare(self, prompt, **kwargs):\n                self.stream.prepare(prompt, **kwargs)\n            def update_prompt(self, prompt):\n                self.stream.update_prompt(prompt)\n            def update_strength(self, strength: float):\n                new_t_list = get_t_list_for_steps(current_steps, \"img2img\", strength)\n                if getattr(self.stream, \"t_list\", None) != new_t_list:\n                    self.stream.t_list = new_t_list\n                    self.stream.prepare(prompt=current_prompt_a, num_inference_steps=50)\n                    logger.info(f\"Updated img2img denoise strength to {strength:.2f} (t_index_list={new_t_list})\")\n            def __call__(self, image):\n                return self.stream(image)\n\n        stream_img2img = SharedImg2ImgWrapper(stream_core)\n    except Exception as e:\n        print(\">>> Fallback to standalone img2img wrapper:\", e)\n        stream_img2img = StreamDiffusionWrapper(\n            model_id_or_path=\"stabilityai/sd-turbo\",\n            t_index_list=t_init_img,\n            frame_buffer_size=1,\n            width=512,\n            height=512,\n            warmup=5,\n            acceleration=\"none\",\n            mode=\"img2img\",\n            output_type=\"pt\",\n            use_lcm_lora=False,\n            use_tiny_vae=True,\n            cfg_type=\"none\",\n            use_denoising_batch=True,\n            device=device,\n            seed=current_seed,\n        )\n\n    print(\">>> Preparing both pipelines with initial prompt...\")\n    stream_txt2img.prepare(prompt=current_prompt_a, num_inference_steps=50)\n    stream_img2img.prepare(prompt=current_prompt_a, num_inference_steps=50)\n\n    print(\">>> Warming up models...\")\n    with torch.no_grad():\n        for _ in range(4):\n            stream_txt2img()\n        dummy_img = Image.new(\"RGB\", (512, 512), color=(128, 128, 128))\n        for _ in range(3):\n            stream_img2img(dummy_img)\n\n    pipeline_ready = True\n    print(\">>> StreamDiffusion pipeline is READY! <<<\", flush=True)\n\n@app.on_event(\"startup\")\nasync def startup_event():\n    loop = asyncio.get_event_loop()\n    await loop.run_in_executor(None, load_pipeline)\n\n@app.get(\"/\")\ndef index():\n    return HTMLResponse(\"\"\"\n    <html>\n        <head><title>StreamDiffusion TouchDesigner Server</title></head>\n        <body style=\"font-family: sans-serif; padding: 40px; background: #121212; color: #fff;\">\n            <h2>StreamDiffusion (SD-Turbo) Colab Server</h2>\n            <p>Status: <span style=\"color: #0f0;\">Running</span></p>\n            <p>Connect your TouchDesigner client via the WebSocket endpoint: <code>/ws/stream</code></p>\n        </body>\n    </html>\n    \"\"\")\n\n@app.websocket(\"/ws/stream\")\nasync def websocket_stream_endpoint(websocket: WebSocket):\n    global current_prompt_a, current_prompt_b, current_prompt_blend, current_neg_prompt\n    global current_mode, current_seed, current_strength\n    await websocket.accept()\n    logger.info(\"TouchDesigner client connected to WebSocket.\")\n\n    vram_s, temp_s = get_gpu_telemetry()\n    await websocket.send_text(json.dumps({\n        \"type\": \"welcome\",\n        \"message\": \"StreamDiffusion server connected\",\n        \"engine\": active_engine,\n        \"steps\": current_steps,\n        \"prompt_a\": current_prompt_a,\n        \"prompt_b\": current_prompt_b,\n        \"blend\": current_prompt_blend,\n        \"mode\": current_mode,\n        \"seed\": current_seed,\n        \"strength\": current_strength,\n        \"vram\": vram_s,\n        \"temp\": temp_s,\n    }))\n\n    CHUNK_SIZE = 8192\n    frame_counter = 0\n\n    async def send_chunked_frame(ws: WebSocket, jpg_bytes: bytes):\n        nonlocal frame_counter\n        frame_counter = (frame_counter + 1) % 256\n        total_chunks = (len(jpg_bytes) + CHUNK_SIZE - 1) // CHUNK_SIZE\n        for chunk_idx in range(total_chunks):\n            start = chunk_idx * CHUNK_SIZE\n            end = min(start + CHUNK_SIZE, len(jpg_bytes))\n            chunk = jpg_bytes[start:end]\n            header = bytes([frame_counter, chunk_idx, total_chunks])\n            try:\n                await ws.send_bytes(header + chunk)\n            except Exception:\n                break\n\n    txt2img_task = None\n    stop_event = asyncio.Event()\n    img2img_counter = 0\n    td_chunk_buffers = {}\n\n    async def txt2img_loop():\n        target_fps = 25\n        frame_delay = 1.0 / target_fps\n        frame_count = 0\n        logger.info(f\"txt2img_loop started! (pipeline_ready={pipeline_ready}, current_mode={current_mode})\")\n        while not stop_event.is_set():\n            start_time = time.time()\n            if pipeline_ready and current_mode == \"txt2img\":\n                try:\n                    loop = asyncio.get_event_loop()\n                    t0 = time.time()\n                    output_tensor = await loop.run_in_executor(None, run_model_inference)\n                    inf_ms = (time.time() - t0) * 1000\n\n                    if output_tensor is not None:\n                        arr_bgr = tensor_to_bgr(output_tensor)\n                        jpg_bytes = compress_frame(arr_bgr)\n                        await send_chunked_frame(websocket, jpg_bytes)\n\n                        frame_count += 1\n                        if frame_count % 10 == 0:\n                            try:\n                                v_str, t_str = get_gpu_telemetry()\n                                await websocket.send_text(json.dumps({\n                                    \"type\": \"stats\",\n                                    \"engine\": active_engine,\n                                    \"steps\": current_steps,\n                                    \"inf_ms\": round(inf_ms, 1),\n                                    \"vram\": v_str,\n                                    \"temp\": t_str,\n                                    \"mode\": current_mode,\n                                    \"strength\": current_strength\n                                }))\n                            except Exception:\n                                pass\n\n                        if frame_count % 25 == 1:\n                            logger.info(f\"StreamDiffusion active: frame #{frame_count} ({inf_ms:.1f}ms, {len(jpg_bytes)} bytes)\")\n                except Exception as e:\n                    if not stop_event.is_set():\n                        logger.error(f\"Inference error in txt2img: {traceback.format_exc()}\")\n\n            elapsed = time.time() - start_time\n            delay = max(0.005, frame_delay - elapsed)\n            await asyncio.sleep(delay)\n\n    txt2img_task = asyncio.create_task(txt2img_loop())\n\n    try:\n        while True:\n            message = await websocket.receive()\n            if message.get(\"type\") == \"websocket.disconnect\":\n                logger.info(\"TouchDesigner client disconnected normally.\")\n                break\n\n            if \"text\" in message:\n                try:\n                    data = json.loads(message[\"text\"])\n                    msg_type = data.get(\"type\", \"\")\n\n                    if msg_type == \"control\" or \"prompt\" in data or \"mode\" in data or \"seed\" in data or \"strength\" in data or \"blend\" in data:\n                        p_a = data.get(\"prompt_a\") or data.get(\"prompt\")\n                        p_b = data.get(\"prompt_b\", \"\")\n                        blend_val = data.get(\"blend\", 0.0)\n                        neg = data.get(\"neg_prompt\", \"\")\n\n                        new_mode = data.get(\"mode\")\n                        new_seed = data.get(\"seed\")\n                        new_strength = data.get(\"strength\")\n\n                        apply_prompts_and_blend(p_a, p_b, blend_val, neg)\n\n                        if new_mode is not None:\n                            mode_str = str(new_mode).lower()\n                            if \"txt\" in mode_str or \"text\" in mode_str:\n                                current_mode = \"txt2img\"\n                            elif \"img\" in mode_str or \"webcam\" in mode_str or \"v2v\" in mode_str or \"noise\" in mode_str:\n                                current_mode = \"img2img\"\n                            else:\n                                current_mode = \"img2img\"\n                            logger.info(f\"Switching mode to: {current_mode}\")\n\n                        if new_seed is not None and new_seed != current_seed:\n                            current_seed = new_seed\n                            set_seed(current_seed)\n\n                        if new_strength is not None and abs(float(new_strength) - current_strength) > 0.01:\n                            apply_strength(float(new_strength))\n\n                        new_steps = data.get(\"steps\")\n                        if new_steps is not None and int(new_steps) != current_steps:\n                            apply_steps(new_steps)\n\n                    elif msg_type == \"ping\":\n                        await websocket.send_text(json.dumps({\"type\": \"pong\", \"time\": time.time()}))\n\n                except Exception as e:\n                    logger.error(f\"Error handling text message: {e}\")\n\n            elif \"bytes\" in message:\n                if pipeline_ready and current_mode in (\"img2img\", \"v2v\"):\n                    raw = message[\"bytes\"]\n                    try:\n                        frame_bytes = None\n                        if raw.startswith(b\"FRM:\") and len(raw) >= 7:\n                            frame_id = raw[4]\n                            chunk_idx = raw[5]\n                            total_chunks = raw[6]\n                            chunk_data = raw[7:]\n\n                            if frame_id not in td_chunk_buffers:\n                                td_chunk_buffers.clear()\n                                td_chunk_buffers[frame_id] = [None] * total_chunks\n\n                            if chunk_idx < total_chunks:\n                                td_chunk_buffers[frame_id][chunk_idx] = chunk_data\n\n                            if all(c is not None for c in td_chunk_buffers[frame_id]):\n                                frame_bytes = b\"\".join(td_chunk_buffers[frame_id])\n                                del td_chunk_buffers[frame_id]\n                        elif raw.startswith(b\"\\xff\\xd8\"):\n                            frame_bytes = raw\n                        else:\n                            frame_bytes = raw\n\n                        if frame_bytes is not None:\n                            nparr = np.frombuffer(frame_bytes, np.uint8)\n                            in_bgr = cv2.imdecode(nparr, cv2.IMREAD_COLOR)\n                            if in_bgr is not None:\n                                in_rgb = cv2.cvtColor(in_bgr, cv2.COLOR_BGR2RGB)\n                                in_pil = Image.fromarray(in_rgb).resize((512, 512))\n\n                                loop = asyncio.get_event_loop()\n                                t0 = time.time()\n                                output = await loop.run_in_executor(None, run_model_inference, in_pil)\n                                inf_ms = (time.time() - t0) * 1000\n\n                                out_bgr = tensor_to_bgr(output)\n                                out_bytes = compress_frame(out_bgr)\n                                await send_chunked_frame(websocket, out_bytes)\n\n                                img2img_counter += 1\n                                if img2img_counter % 10 == 0:\n                                    try:\n                                        v_str, t_str = get_gpu_telemetry()\n                                        await websocket.send_text(json.dumps({\n                                            \"type\": \"stats\",\n                                            \"engine\": active_engine,\n                                            \"steps\": current_steps,\n                                            \"inf_ms\": round(inf_ms, 1),\n                                            \"vram\": v_str,\n                                            \"temp\": t_str,\n                                            \"mode\": current_mode,\n                                            \"strength\": current_strength\n                                        }))\n                                    except Exception:\n                                        pass\n\n                                if img2img_counter % 25 == 1:\n                                    logger.info(f\"StreamDiffusion img2img: frame #{img2img_counter} ({inf_ms:.1f}ms, {len(out_bytes)} bytes)\")\n                    except Exception as e:\n                        if not stop_event.is_set():\n                            logger.error(f\"Error in img2img processing: {traceback.format_exc()}\")\n\n    except WebSocketDisconnect:\n        logger.info(\"TouchDesigner client disconnected gracefully.\")\n    except Exception as e:\n        logger.error(f\"WebSocket error: {e}\")\n    finally:\n        stop_event.set()\n        if txt2img_task:\n            txt2img_task.cancel()\n            try:\n                await txt2img_task\n            except asyncio.CancelledError:\n                pass\n\nif __name__ == \"__main__\":\n    import uvicorn\n    uvicorn.run(app, host=\"0.0.0.0\", port=8000, log_level=\"info\", ws_per_message_deflate=False)\n"

with open('server.py', 'w') as f:
    f.write(server_code.strip())
print('✅ server.py written successfully!')

In [ ]:
# @title Step 4: Launch Server & Start Cloudflare Tunnel
# @markdown ---
# @markdown ### ⚡ 1-Toggle Acceleration Switch:
# @markdown - **Unchecked (Default)**: Fast 35-second boot (PyTorch FP16, ~75ms inference).
# @markdown - **Checked**: TensorRT Turbo mode (~25-35ms inference, ~10-15 min first-time compile).
ENABLE_TENSORRT = False # @param {type:"boolean"}
# @markdown ---

import os
import subprocess
import time
import sys
import re
from IPython.display import display, HTML

# Kill any lingering process on port 8000
os.system('fuser -k 8000/tcp 2>/dev/null || true')

server_cmd = [sys.executable, 'server.py']
if ENABLE_TENSORRT:
    print('⚡ TensorRT Acceleration Enabled!')
    print('   Checking TensorRT tools (first run compiles engines in ~10-15 mins)...')
    os.system('python -m streamdiffusion.tools.install-tensorrt || true')
    server_cmd.append('--tensorrt')
else:
    print('⚡ Standard PyTorch Native Mode (Fast 35s boot)...')

print('Starting Python server and downloading SD-Turbo weights (~2.5GB)...')
server_proc = subprocess.Popen(
    server_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# Stream logs until model is fully ready
ready = False
for line in server_proc.stdout:
    print(line, end='', flush=True)
    if 'READY!' in line or 'Application startup complete' in line or 'Uvicorn running' in line:
        ready = True
        break
    if server_proc.poll() is not None:
        break

if ready:
    print('\nStarting Cloudflare Tunnel to expose port 8000...')
    tunnel_proc = subprocess.Popen(
        ['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    tunnel_url = None
    for line in tunnel_proc.stdout:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            break

    if tunnel_url:
        ws_url = tunnel_url.replace('https://', 'wss://') + '/ws/stream'
        display(HTML(f'''
        <div style="background:#1a2e1a; border:2px solid #00ff88; padding:20px; border-radius:10px; margin:20px 0;">
            <h2 style="color:#00ff88; margin:0 0 10px 0;">🚀 StreamDiffusion Server is READY & STREAMING!</h2>
            <p style="color:#eee; font-size:16px; margin:0 0 10px 0;">Copy this URL and paste it into TouchDesigner:</p>
            <div style="background:#000; padding:12px; border-radius:6px; font-family:monospace; font-size:18px; color:#58a6ff;">
                {ws_url}
            </div>
        </div>
        '''))
else:
    print('\n❌ Server failed to start. Review error logs above.')

# Keep server running
try:
    while True:
        line = server_proc.stdout.readline()
        if line:
            print(line, end='', flush=True)
        time.sleep(0.01)
except KeyboardInterrupt:
    print('Stopping server...')
    server_proc.terminate()
    if 'tunnel_proc' in locals():
        tunnel_proc.terminate()
